In [1]:
import nibabel as nib
from nibabel.freesurfer.io import read_geometry 
import matplotlib.pyplot as plt
import numpy as np
from nilearn.image import resample_to_img


In [9]:
def get_nifti_metadata(file_path):
    """Extract metadata from a NIfTI file."""
    img = nib.load(file_path)
    header = img.header
    metadata = {
        "dim": header["dim"],  # Image dimensions
        "datatype": header["datatype"],  # Data type
        "vox_offset": header["vox_offset"],  # Offset into data
        "pixdim": header["pixdim"],  # Voxel size
        "sform_code": header["sform_code"],  # Sform transformation
        "qform_code": header["qform_code"],  # Qform transformation
        "affine": img.affine,  # Affine transformation matrix
    }
    return metadata

def compare_metadata(file1, file2):
    """Compare metadata of two NIfTI files and print differences."""
    meta1 = get_nifti_metadata(file1)
    meta2 = get_nifti_metadata(file2)

    print("\nComparing Metadata...\n")
    for key in meta1:
        if (meta1[key] != meta2[key]).any():
            print(f"Difference in {key}:")
            print(f"  File 1: {meta1[key]}")
            print(f"  File 2: {meta2[key]}\n")

# Example usage
file1_path = "./mri_segs/dhcp/sub-CC00060XX03_ses-12501_synthseg_robust_parc_cpu_mac.nii.gz"
file2_path = "./mri_segs/bobs/sub-116056_ses-3mo_synthseg_robust_parc_cpu_mac.nii.gz"

compare_metadata(file1_path, file2_path)



Comparing Metadata...

Difference in dim:
  File 1: [  3 145 145 102   1   1   1   1]
  File 2: [  3 182 218 182   1   1   1   1]

Difference in pixdim:
  File 1: [-1.  1.  1.  1.  1.  1.  1.  1.]
  File 2: [-1.  1.  1.  1.  0.  1.  1.  1.]

Difference in sform_code:
  File 1: 2
  File 2: 4

Difference in qform_code:
  File 1: 0
  File 2: 4

Difference in affine:
  File 1: [[ -1.           0.          -0.          62.68331528]
 [  0.           1.          -0.         -54.17528534]
 [  0.           0.           1.         -35.15189362]
 [  0.           0.           0.           1.        ]]
  File 2: [[  -1.    0.    0.   90.]
 [   0.    1.    0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]



In [13]:
aseg = "/Users/arnaud/Downloads/bobs_mri/sub-116056/ses-3mo/anat/sub-116056_ses-3mo_space-INFANTMNIacpc_desc-aseg_dseg.nii.gz"
t1 = "/Users/arnaud/Downloads/bobs_mri/sub-116056/ses-3mo/anat/sub-116056_ses-3mo_space-INFANTMNIacpc_T1w.nii.gz"
compare_metadata(aseg, t1)


Comparing Metadata...

Difference in datatype:
  File 1: 512
  File 2: 16

Difference in sform_code:
  File 1: 0
  File 2: 4

Difference in qform_code:
  File 1: 1
  File 2: 4



In [14]:
(np.unique(nib.load(aseg).get_fdata()))


array([  0.,   2.,   3.,   4.,   5.,   7.,   8.,  10.,  11.,  12.,  13.,
        14.,  15.,  16.,  17.,  18.,  26.,  28.,  31.,  41.,  42.,  43.,
        44.,  46.,  47.,  49.,  50.,  51.,  52.,  53.,  54.,  58.,  60.,
        63., 172.])

In [16]:
!pip install antspyx  # Python wrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 8.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 26.9 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip install --upgrade pip


## Registration attempt

In [2]:
import ants

In [2]:
# Load input image (your NIfTI file)
dhcp_1 = "/Users/arnaud/Downloads/sourcedata_orig/sub-CC00060XX03/ses-12501/anat/sub-CC00060XX03_ses-12501_T1w.nii.gz"
input_image = ants.image_read(dhcp_1)


In [4]:
input_image

ANTsImage (RPI)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (290, 290, 203)
	 Spacing    : (0.5, 0.5, 0.5)
	 Origin     : (-62.9333, 54.4253, -35.4019)
	 Direction  : [ 1.  0.  0.  0. -1.  0.  0.  0.  1.]

In [6]:
# Load MNI152 template (commonly available in neuroimaging tools like FSL or ANTs)
temp1 = "./registre/Akiyama_6Month_1_5T.nii.gz"
mni_template = ants.image_read(temp1)
mni_template

ANTsImage (LPI)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (195, 233, 159)
	 Spacing    : (1.0, 1.0, 1.0)
	 Origin     : (96.0, 134.0, -72.0)
	 Direction  : [-1.  0.  0.  0. -1.  0.  0.  0.  1.]

In [7]:
# Perform registration (SyN algorithm: Symmetric Normalization)
transformed_image = ants.registration(fixed=mni_template, moving=input_image, type_of_transform='SyN')

# Save the transformed image
ants.image_write(transformed_image['warpedmovout'], "output_registre.nii.gz")

In [12]:
compare_metadata(file2_path, "output_registre.nii.gz")


Comparing Metadata...

Difference in dim:
  File 1: [  3 182 218 182   1   1   1   1]
  File 2: [  3 195 233 159   1   1   1   1]

Difference in datatype:
  File 1: 8
  File 2: 16

Difference in pixdim:
  File 1: [-1.  1.  1.  1.  0.  1.  1.  1.]
  File 2: [1. 1. 1. 1. 0. 0. 0. 0.]

Difference in sform_code:
  File 1: 4
  File 2: 1

Difference in qform_code:
  File 1: 4
  File 2: 1

Difference in affine:
  File 1: [[  -1.    0.    0.   90.]
 [   0.    1.    0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]
  File 2: [[   1.    0.   -0.  -96.]
 [   0.    1.   -0. -134.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]



Registration with T2  ?

In [6]:
# Load input image (your NIfTI file)
dhcp_2 = "/Users/arnaud/Downloads/sourcedata_orig/sub-CC00058XX09/ses-11300/anat/sub-CC00058XX09_ses-11300_T2w.nii.gz"
input_image2 = ants.image_read(dhcp_2)

# Load MNI152 template (commonly available in neuroimaging tools like FSL or ANTs)
temp1 = "./registre/Akiyama_6Month_1_5T.nii.gz"
mni_template = ants.image_read(temp1)


In [7]:
# Perform registration (SyN algorithm: Symmetric Normalization)
transformed_image2 = ants.registration(fixed=mni_template, moving=input_image2, type_of_transform='SyN')

# Save the transformed image
ants.image_write(transformed_image2['warpedmovout'], "output_registre_T2.nii.gz")